# **此教程全程基于ubuntu22.04**
- 请先查看群公告安装ubuntu系统

# **本教程所参考的YOLO教程csdn地址**
- 如本教程有不懂的地方可先查看此教程
- 如仍无法解决可发至QQ群
>https://blog.csdn.net/kushe123/article/details/113702225

# **yolo官方的github地址**

>https://github.com/ultralytics/yolov5

# **一、环境安装**

- 安装NVIDIA驱动，需要NVIDIA显卡
- 此教程使用RTX4060显卡
    - 根据自己电脑显卡型号安装对应显卡驱动
    - [NVIDIA显卡的Ubuntu驱动程序安装方法-哔哩哔哩](https://b23.tv/sqn2Z4M)

- 安装好后输入`nvidia-smi`应输出如下图所示信息

![图片.png](03.png)

- `anaconda`安装可直接询问deepseek: ubuntu 22.04如何安装anaconda
    - 或者参考B站视频：
    - [Anaconda安装【Linux系统】-哔哩哔哩](https://b23.tv/PaTsyAf)

1. conda创建yolo环境
>```bash
>#需要预先安装anaconda
>conda create -n yolo_v5 python=3.10
>#进入环境
>conda activate yolo_v5
>```


- 进入后用户名前面会有环境名即（yolo_v5）
    - 如果新开终端需要重新进入环境

![图片.png](01.png)

2. 安装`pytorch`
- 需要NVIDIA显卡并安装对应cuda版本
>```bash
>#检查当前pip是否是环境内的pip:应输出下图路径
>which pip
>#使用pip安装pytorch
>#需要先查看cuda版本安装对应版本pytorch
>#此版本cuda仅在4060能够正常使用，其他显卡类型不一定使用此版本
>pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
>```

![图片.png](02.png)
- cuda安装完成验证：

![图片.png](05.png)

3. 安装`jupyter lab`
>```bash
>pip install jupyterlab
>#可能会有其他依赖未安装
>```

4. 拉取代码
- 此步可能会因为官方仓库更新导致环境依赖与当前教程不同等问题
- 直接拉取当前仓库2026.3.8版本
>```bash
>git clone https://github.com/wei-a11y/yolov5_master.git
>cd yolov5_master
>pip install -r requirements.txt  # 安装依赖项
>```

5. 安装`tensorboard`
>```bash
>pip install tensorboard
>```

6. 下载依赖包
>```bash
># 在yolov5目录下下载
>pip install -r requirements.txt
>```

# **二、使用`labelimg`制作数据集**

## **2.1 创建`labelimg`专属`Conda`环境**

>```bash
># 创建名为 labelimg 的环境，指定 python 版本
>conda create -n labelimg python=3.8 -y
># 激活环境
>conda activate labelimg
>```

## **2.2 安装依赖并安装 `labelImg`**

>```bash
># 安装 PyQt5 和 lxml
>conda install pyqt=5
>conda install lxml
># 使用 pip 安装 labelImg
>pip install labelImg
>```

## **2.3 数据准备**

### **2.3.1 创建文件夹**

VOC2007的目录结构为：

├── VOC2007  
│├── JPEGImages  存放需要打标签的图片文件  
│├── Annotations  存放标注的标签文件  
│├── predefined_classes.txt  定义自己要标注的所有类别

### **2.3.2 标注前的设置**

- 在`JPEGImages`这个文件夹放置待标注的图片

![图片.png](1.png)

- 在`predefined_classes.txt` 这个`txt`文档里面输入定义的类别种类

![图片.png](2.png)

## **2.4 启动软件**

>```bash
>#进入到创建的`VOC2007`路径，例如：
>cd yolo_v5/test/copy/VOCdevkit/VOC2007
>#启动`labelimg`
>labelImg JPEGImages predefined_classes.txt
>#意思是打开labelimg工具；打开JPEGImage文件夹，初始化predefined_classes.txt里面定义的类。
>```

![图片.png](3.png)
- 待标注图片数据的路径文件夹，这里输入命令的时候就选定了JPEGImages。

![图片.png](4.png)
- 保存类别标签的路径文件夹，这里我们选定了Annotations文件夹。

![图片.png](5.png)
- 这个按键可以说明我们标注的标签为voc格式，点击可以换成yolo或者createML格式。

![图片.png](6.png)
- 点击View，会出现如图红色框框中的选项。最好和我一样把勾勾勾上。

- 常用快捷键
    - A：切换到上一张图片
    - D：切换到下一张图片
    - W：调出标注十字架
    - del ：删除标注框框
    - Ctrl+u：选择标注的图片文件夹
    - Ctrl+r：选择标注好的label标签存在的文件夹

## **2.5 标注**

- 选定我们需要标注的对象。按住鼠标左键拖出框框就可以

![图片.png](7.png)

- 当我们选定目标以后，就会加载出来predefined_classes.txt  定义自己要标注的所有类别

- 当标注错需要修改时点击左侧`Edit RectBox`

![图片.png](8.png)

- 然后在右上角选中需要修改的标签
- 再按右键即可编辑

![图片.png](9.png)

- 标签打完以后可以去Annotations 文件下看到标签文件已经保存在这个目录下。

![图片.png](10.png)

# **三、 将数据集划分为数据集和验证集**

- 标注的是`VOC`格式，而`yolov5`训练所需要的文件格式是`yolo(txt格式)`的
- 这里就需要对`xml`格式的标签文件转换为`txt`文件。
- 同时训练自己的yolov5检测 模型 的时候，数据集需要划分为训练集和验证集。

- `transform.py`用于将`xml`格式的标注文件转换为`txt`格式的标注文件
- 并按比例划分为训练集和验证集

- 数据集的格式结构必须严格按照如图的样式来，因为代码已经将文件名写死了

![图片.png](11.png)

- classes里面必须正确填写xml里面已经标注好的类

![图片.png](12.png)

- 将代码和数据在同一目录下运行，

![图片.png](04.png)
- 得到如下的结果

![图片.png](13.png)

- 划分结束后将`VOCdevkit`整个文件夹放到`yolov5`的代码中（原本是没有VOCdevkit文件夹，直接将上面的文件夹复制到yolov5-master文件夹下就行）

![图片.png](14.png)

# **四、 训练**

## **4.1 数据配置文件**

- 在`data`下新建`my_dataset.yaml`

![图片.png](15.png)

- 内容：

>```yaml
># Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/>imgs1, path/to/imgs2, ..]
>path: /home/www/yolo_v5/yolov5/VOCdevkit  # 数据集根目录
>train: images/train  # 训练数据集 (relative to 'path') 
>val: images/val  # 验证集 (relative to 'path') 
>test:  # 测试数据集可不填 (optional)
>
># Classes
>nc: 2  # 识别的类别数
>names: # 类别名
>  0: box
>  1: gauze
>```

- 类别数和类别名需要与标注时设定的数量和类别名一致

## **4.2 开始训练**

- 终端进入`yolo`环境，输入训练命令即可开始训练

>```bash
>python3 train.py --img 640 --batch 32 --epochs 200 --data my_dataset.yaml --weights yolov5s.pt --name yolov5s_test_3.6
>```

- 相关配置参数可在train.py第568-612行更改默认值
- 根据自己电脑性能调整相关训练参数
- img：图像大小
- batch： 批次大小
- epochs：训练轮数
- data：4.1中创建的配置文件名称
- weights：预训练权重
- name：训练名称（保存在runs/train目录下）

## **4.3 使用`tensorboard`查看训练结果**

>```bash
>tensorboard --logdir=runs/train
>```

# **五、 推理测试**

1. 找到你训练好的权重文件路径，例如： `runs/train/yolov5s_test_3.6/weights/best.pt`
2. 找一张图片（例如 `data/images/vla_test.jpg`），运行推理

>```bash
>python detect.py --weights runs/train/yolov5s_test_3.6/weights/best.pt --source data/images/val_test.jpg --device 0
># 使用默认摄像头（通常是 0）
>python detect.py --weights runs/train/yolov5s_test_3.6/weights/best.pt --source 0
>```

![图片.png](16.png)

- 推理结果在最后一行`Results`

![图片.png](17.png)

- 使用摄像头

![图片.png](18.png)